In [17]:
# =====================================================
# STEP 1: Import Libraries
# =====================================================

import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    f1_score
)

from sklearn.feature_selection import (
    SelectKBest,
    chi2,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier

from scipy.sparse import hstack

Load Dataset

In [20]:
import pandas as pd

df = pd.read_csv(
    "Amazon_Reviews.csv",
    engine="python",
    encoding="latin1",
    on_bad_lines="skip"
)

In [21]:
print(df.head())

      Reviewer Name                     Profile Link Country Review Count  \
0        Eugene ath  /users/66e8185ff1598352d6b3701a      US     1 review   
1  Daniel ohalloran  /users/5d75e460200c1f6a6373648c      GB    9 reviews   
2          p fisher  /users/546cfcf1000064000197b88f      GB   90 reviews   
3         Greg Dunn  /users/62c35cdbacc0ea0012ccaffa      AU    5 reviews   
4     Sheila Hannah  /users/5ddbe429478d88251550610e      GB    8 reviews   

                Review Date                  Rating  \
0  2024-09-16T13:44:26.000Z  Rated 1 out of 5 stars   
1  2024-09-16T18:26:46.000Z  Rated 1 out of 5 stars   
2  2024-09-16T21:47:39.000Z  Rated 1 out of 5 stars   
3  2024-09-17T07:15:49.000Z  Rated 1 out of 5 stars   
4  2024-09-16T18:37:17.000Z  Rated 1 out of 5 stars   

                                      Review Title  \
0       A Store That Doesn't Want to Sell Anything   
1         Had multiple orders one turned up andâ¦   
2                      I informed these repr

In [22]:
print(df.columns.tolist())

['Reviewer Name', 'Profile Link', 'Country', 'Review Count', 'Review Date', 'Rating', 'Review Title', 'Review Text', 'Date of Experience']


In [23]:
print(df.shape)

(21214, 9)


In [28]:
df = df[['Review Text', 'Rating']]

In [29]:
df.dropna(inplace=True)

df.head()

,Review Text,Rating
0,"I registered on the website, tried to order a ...",Rated 1 out of 5 stars
1,Had multiple orders one turned up and driver h...,Rated 1 out of 5 stars
2,I informed these reprobates that I WOULD NOT B...,Rated 1 out of 5 stars
3,I have bought from Amazon before and no proble...,Rated 1 out of 5 stars
4,If I could give a lower rate I would! I cancel...,Rated 1 out of 5 stars


 Convert Ratings into Sentiment Classes

In [32]:


df["Rating"] = (
    df["Rating"]
    .astype(str)
    .str.extract(r'(\d+\.?\d*)')[0]
)

df["Rating"] = pd.to_numeric(
    df["Rating"],
    errors='coerce'
)

df.dropna(subset=["Rating"], inplace=True)

df["Rating"] = df["Rating"].astype(int)

print(df["Rating"].unique())

[1 5 2 4 3]


 Text Cleaning

In [34]:
def clean_text(text):

    text=str(text)

    text=text.lower()

    text=re.sub(r"http\S+"," ",text)

    text=re.sub(r"[^a-zA-Z ]"," ",text)

    text=re.sub(r"\s+"," ",text)

    return text.strip()


df["Clean_Text"] = df["Review Text"].apply(clean_text)

Create Linguistic Features

In [36]:


negation_words = [
    "not","never","no",
    "nothing","none",
    "cannot","can't",
    "won't","don't",
    "isn't"
]


def sentence_length(text):

    return len(text.split())


def punctuation_density(text):

    punct_count=sum(
        1 for c in text
        if c in string.punctuation
    )

    total=max(len(text),1)

    return punct_count/total


def avg_word_length(text):

    words=text.split()

    if len(words)==0:
        return 0

    return np.mean(
        [len(word)
        for word in words]
    )


def negation_presence(text):

    words=text.split()

    return int(
        any(
            word in negation_words
            for word in words
        )
    )


df["sentence_length"]=df["Review Text"].apply(sentence_length)

df["punct_density"]=df["Review Text"].apply(punctuation_density)

df["avg_word_length"]=df["Review Text"].apply(avg_word_length)

df["negation"]=df["Review Text"].apply(negation_presence)

df.head()

,Review Text,Rating,Clean_Text,sentence_length,punct_density,avg_word_length,negation
0,"I registered on the website, tried to order a ...",1,i registered on the website tried to order a l...,106,0.040678,4.575472,1
1,Had multiple orders one turned up and driver h...,1,had multiple orders one turned up and driver h...,53,0.020478,4.547170,1
2,I informed these reprobates that I WOULD NOT B...,1,i informed these reprobates that i would not b...,122,0.016207,4.065574,1
3,I have bought from Amazon before and no proble...,1,i have bought from amazon before and no proble...,82,0.017778,4.487805,1
4,If I could give a lower rate I would! I cancel...,1,if i could give a lower rate i would i cancell...,100,0.018587,4.380000,1


TF-IDF Features

In [37]:
tfidf=TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_text=tfidf.fit_transform(
    df["Clean_Text"]
)

print(X_text.shape)

(21055, 5000)


Create sentiment labels again

In [39]:
def create_sentiment(rating):

    if rating <= 2:
        return "Negative"

    elif rating == 3:
        return "Neutral"

    else:
        return "Positive"


df["Sentiment"] = df["Rating"].apply(create_sentiment)

print(df["Sentiment"].value_counts())

print(df.columns.tolist())

Sentiment
Negative    14350
Positive     5820
Neutral       885
Name: count, dtype: int64
['Review Text', 'Rating', 'Clean_Text', 'sentence_length', 'punct_density', 'avg_word_length', 'negation', 'Sentiment']


Combine TF-IDF + Linguistic Features

In [40]:
extra_features=df[
    [
        "sentence_length",
        "punct_density",
        "avg_word_length",
        "negation"
    ]
].values


X=hstack(
    [X_text,extra_features]
)

y=df["Sentiment"]

 Encode Labels

In [41]:
encoder=LabelEncoder()

y=encoder.fit_transform(y)

print(
encoder.classes_
)

['Negative' 'Neutral' 'Positive']


Train Test Split

In [42]:
X_train,X_test,y_train,y_test=(
train_test_split(
    X,
    y,
    test_size=.2,
    random_state=42,
    stratify=y
)
)

Feature Selection

Chi-square

In [43]:
selector_chi=SelectKBest(
    score_func=chi2,
    k=3000
)

X_train_chi=selector_chi.fit_transform(
    X_train,
    y_train
)

X_test_chi=selector_chi.transform(
    X_test
)

Mutual Information

In [44]:
selector_mi=SelectKBest(
    score_func=mutual_info_classif,
    k=3000
)

X_train_mi=selector_mi.fit_transform(
    X_train.toarray(),
    y_train
)

X_test_mi=selector_mi.transform(
    X_test.toarray()
)

Models

In [45]:


models={

"Logistic Regression":
LogisticRegression(
    max_iter=1000
),

"Multinomial NB":
MultinomialNB(),

"Linear SVM":
LinearSVC(),

"Gradient Boosting":
GradientBoostingClassifier(
    n_estimators=100
)

}

Train and Evaluate

In [46]:


results=[]

for name,model in models.items():

    print("\n")
    print("="*50)
    print(name)

    if name=="Gradient Boosting":

        model.fit(
            X_train_chi.toarray(),
            y_train
        )

        pred=model.predict(
            X_test_chi.toarray()
        )

    else:

        model.fit(
            X_train_chi,
            y_train
        )

        pred=model.predict(
            X_test_chi
        )



    macro=f1_score(
        y_test,
        pred,
        average="macro"
    )

    print(
        "Macro F1:",
        round(macro,4)
    )

    print(
        classification_report(
            y_test,
            pred,
            target_names=
            encoder.classes_
        )
    )

    results.append(
        [name,macro]
    )



Logistic Regression


C:\Users\gayat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\gayat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramet

Macro F1: 0.6065
              precision    recall  f1-score   support

    Negative       0.92      0.96      0.94      2870
     Neutral       0.12      0.01      0.01       177
    Positive       0.86      0.88      0.87      1164

    accuracy                           0.90      4211
   macro avg       0.63      0.62      0.61      4211
weighted avg       0.87      0.90      0.88      4211



Multinomial NB
Macro F1: 0.5235
              precision    recall  f1-score   support

    Negative       0.90      0.82      0.86      2870
     Neutral       0.00      0.00      0.00       177
    Positive       0.62      0.85      0.71      1164

    accuracy                           0.79      4211
   macro avg       0.51      0.56      0.52      4211
weighted avg       0.78      0.79      0.78      4211



Linear SVM
Macro F1: 0.6235
              precision    recall  f1-score   support

    Negative       0.92      0.96      0.94      2870
     Neutral       0.36      0.03      0.05     

Compare Models

In [47]:
result_df=pd.DataFrame(
    results,
    columns=[
        "Model",
        "Macro_F1"
    ]
)

print(
result_df.sort_values(
    by="Macro_F1",
    ascending=False
)
)

                 Model  Macro_F1
2           Linear SVM  0.623456
0  Logistic Regression  0.606479
3    Gradient Boosting  0.588566
1       Multinomial NB  0.523514


Top Discriminating Features

In [48]:


best_model=LogisticRegression(
    max_iter=1000
)

best_model.fit(
    X_train_chi,
    y_train
)

feature_names=np.array(
    tfidf.get_feature_names_out()
)

selected=selector_chi.get_support()

selected_features=feature_names[
selected[:len(feature_names)]
]

for i,class_name in enumerate(
    encoder.classes_
):

    print("\n")

    print(
    "Top words for:",
    class_name
    )

    top=np.argsort(
        best_model.coef_[i]
    )[-10:]

    print(
        selected_features[top]
    )



Top words for: Negative
['will' 'useless' 'account' 'again' 'not' 'terrible' 'now' 'horrible'
 'poor' 'worst']


Top words for: Neutral
['products' 'okay' 'sometimes' 'amazon amazon' 'like' 'some' 'amazon but'
 'good' 'however' 'but']


Top words for: Positive
['perfect' 'everything' 'fast' 'quickly' 'best' 'always' 'amazing' 'love'
 'excellent' 'great']


C:\Users\gayat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [49]:
import joblib

# Save trained model
joblib.dump(best_model, "sentiment_model.pkl")

# Save TF-IDF vectorizer
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

# Save label encoder
joblib.dump(encoder, "label_encoder.pkl")

# Save Chi-Square selector
joblib.dump(selector_chi, "chi_selector.pkl")

print("Model and files saved successfully!")

Model and files saved successfully!
